In [2]:
import polars as pl

employees = pl.DataFrame({
    "emp_id": [1, 2, 3, 4],
    "name": ["Aman", "Rahul", "Priya", "Neha"]
})

departments = pl.DataFrame({
    "emp_id": [1, 2, 3],
    "dept_name": ["IT", "HR", "CS"]
})

joined_df = employees.join(departments, on="emp_id", how="inner")

print(joined_df)

shape: (3, 3)
┌────────┬───────┬───────────┐
│ emp_id ┆ name  ┆ dept_name │
│ ---    ┆ ---   ┆ ---       │
│ i64    ┆ str   ┆ str       │
╞════════╪═══════╪═══════════╡
│ 1      ┆ Aman  ┆ IT        │
│ 2      ┆ Rahul ┆ HR        │
│ 3      ┆ Priya ┆ CS        │
└────────┴───────┴───────────┘


In [1]:
import polars as pl
import numpy as np

employees = pl.DataFrame({
    "emp_id": [1, 2, 3, 4, 5, 2],
    "name": ["Aman", "Rahul", "Priya", "Neha", "Rohit", "Rahul"],
    "age": ["23", "25", None, "28", "24", "25"]
})

departments = pl.DataFrame({
    "emp_id": [1, 2, 3, 5],
    "dept_name": ["IT", "HR", "IT", "Marketing"]
})

salaries = pl.DataFrame({
    "emp_id": [1, 2, 3, 4],
    "salary": [40000, 60000, 35000, 80000]
})

employees = employees.unique()
employees = employees.with_columns(pl.col("age").fill_null("0").cast(pl.Int64))

df = employees.join(departments, on="emp_id", how="left")
df = df.join(salaries, on="emp_id", how="left")

df = df.with_columns([
    pl.col("dept_name").fill_null("Unassigned"),
    pl.col("salary").fill_null(0)
])

new_hires = pl.DataFrame({
    "emp_id": [6, 7],
    "name": ["Saurabh", "Vijay"],
    "age": [26, 22],
    "dept_name": ["IT", "Sales"],
    "salary": [45000, 42000]
})

df = pl.concat([df, new_hires])

df = df.with_columns((pl.col("salary") * 0.10).alias("Bonus"))

summary = df.group_by("dept_name").agg([
    pl.col("salary").mean().alias("Avg_Salary"),
    pl.len().alias("Headcount")
])

print(df)
print(summary)

df.write_csv("final_employee_pipeline.csv")

shape: (7, 6)
┌────────┬─────────┬─────┬────────────┬────────┬────────┐
│ emp_id ┆ name    ┆ age ┆ dept_name  ┆ salary ┆ Bonus  │
│ ---    ┆ ---     ┆ --- ┆ ---        ┆ ---    ┆ ---    │
│ i64    ┆ str     ┆ i64 ┆ str        ┆ i64    ┆ f64    │
╞════════╪═════════╪═════╪════════════╪════════╪════════╡
│ 4      ┆ Neha    ┆ 28  ┆ Unassigned ┆ 80000  ┆ 8000.0 │
│ 1      ┆ Aman    ┆ 23  ┆ IT         ┆ 40000  ┆ 4000.0 │
│ 3      ┆ Priya   ┆ 0   ┆ IT         ┆ 35000  ┆ 3500.0 │
│ 5      ┆ Rohit   ┆ 24  ┆ Marketing  ┆ 0      ┆ 0.0    │
│ 2      ┆ Rahul   ┆ 25  ┆ HR         ┆ 60000  ┆ 6000.0 │
│ 6      ┆ Saurabh ┆ 26  ┆ IT         ┆ 45000  ┆ 4500.0 │
│ 7      ┆ Vijay   ┆ 22  ┆ Sales      ┆ 42000  ┆ 4200.0 │
└────────┴─────────┴─────┴────────────┴────────┴────────┘
shape: (5, 3)
┌────────────┬────────────┬───────────┐
│ dept_name  ┆ Avg_Salary ┆ Headcount │
│ ---        ┆ ---        ┆ ---       │
│ str        ┆ f64        ┆ u32       │
╞════════════╪════════════╪═══════════╡
│ IT         ┆ 400

In [1]:
import polars as pl

df = pl.DataFrame({
    "Date": ["2023-01-01", "2023-01-01", "2023-01-02", "2023-01-02", "2023-01-01", "2023-01-02"],
    "Category": ["Electronics", "Furniture", "Electronics", "Furniture", "Electronics", "Furniture"],
    "Sales": [1000, 200, None, 300, 1500, None],
    "Rating": [4.5, 3.0, 5.0, None, 4.0, 2.5]
})

df = df.with_columns([
    pl.col("Sales").fill_null(pl.col("Sales").mean().over("Category")),
    pl.col("Rating").fill_null(pl.col("Rating").mean().over("Category"))
])

df = df.with_columns(
    (pl.col("Sales") / pl.col("Sales").sum().over("Category")).alias("Category_Share")
)

df = df.with_columns(
    pl.col("Sales").rank(method="dense", descending=True).over("Category").alias("Sales_Rank")
)

category_stats = df.group_by("Category").agg([
    pl.col("Sales").sum().alias("Total_Revenue"),
    pl.col("Sales").mean().alias("Avg_Transaction"),
    pl.col("Rating").mean().alias("Avg_Rating")
])

final_report = df.join(category_stats, on="Category", how="left")

print(final_report)
print(category_stats)

shape: (6, 9)
┌────────────┬────────────┬────────┬────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ Date       ┆ Category   ┆ Sales  ┆ Rating ┆ … ┆ Sales_Rank ┆ Total_Reve ┆ Avg_Transa ┆ Avg_Ratin │
│ ---        ┆ ---        ┆ ---    ┆ ---    ┆   ┆ ---        ┆ nue        ┆ ction      ┆ g         │
│ str        ┆ str        ┆ f64    ┆ f64    ┆   ┆ u32        ┆ ---        ┆ ---        ┆ ---       │
│            ┆            ┆        ┆        ┆   ┆            ┆ f64        ┆ f64        ┆ f64       │
╞════════════╪════════════╪════════╪════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 2023-01-01 ┆ Electronic ┆ 1000.0 ┆ 4.5    ┆ … ┆ 3          ┆ 3750.0     ┆ 1250.0     ┆ 4.5       │
│            ┆ s          ┆        ┆        ┆   ┆            ┆            ┆            ┆           │
│ 2023-01-01 ┆ Furniture  ┆ 200.0  ┆ 3.0    ┆ … ┆ 3          ┆ 750.0      ┆ 250.0      ┆ 2.75      │
│ 2023-01-02 ┆ Electronic ┆ 1250.0 ┆ 5.0    ┆ … ┆ 2          ┆ 3750.0     ┆ 1